# Corrected comparison: both label pools and current Ani

**Conclusion: the selected candidate has not established a clear improvement over current Ani.** It is slightly worse on the 157-label pool and slightly better on the 235-label pool. Current Ani has higher macro F1 on both.

This corrects two omissions in the earlier report: only the 235-label CV was run, and the reference was Ani's older no-LDA/C0.3 model. The earlier phrase “full testing procedure” was too broad, and the larger claimed gain did not compare against current Ani.

## Exact existing testing procedure

- **Original training pool:** all157 visible training labels, five saved repeats of three-fold multilabel-stratified CV, seeds42–46. Each fit has103–106 labels. The78 validation trials are excluded completely from these fits.
- **Training plus validation pool:** all235 visible labels, eight saved repeats of five-fold CV, seeds100–107. Each fit has188 labels.
- Both models use the same fold assignments and the same1,014 originally unlabeled training trials. All learned preprocessing and pseudo-labeling exclude held-out/test inputs. Score sample F1 over all out-of-fold trials within each repeat, then average repeats.
- Candidate settings are frozen. No retuning, new split generation or test-feature fitting was performed for this audit.

## Current comparator and provenance

`git fetch origin ani` confirmed current commit **8a8c3fca70f13b62e989739a0bc1d97d229186e1**. The exact [Ani notebook](https://github.com/anihadagali7/AML-Midterm/blob/8a8c3fca70f13b62e989739a0bc1d97d229186e1/midterm/part1_best_model.ipynb) is archived in `experiments/current_ani_pool_audit/ani_current_source.ipynb`. Its recipe functions are executed directly, with scaling/PCA/GMM refit on the permitted fold inputs.

Current Ani blends logistic regression0.5 (C0.1), cosine neighbors0.3, and shrinkage LDA0.2, followed by its GMM-assisted two-pass self-training. The legacy model has no LDA and uses C0.3. Ani's native notebook uses all embeddings; this comparison applies the same strict feature policy to both models, as required by the repository's active testing policy. It is not a new Kaggle result or a transductive comparison.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
PART1=Path.cwd()
if not (PART1/'data/part1_train.csv').exists():PART1=PART1/'part1'
OUT=PART1/'experiments/current_ani_pool_audit'
summary=pd.read_csv(OUT/'summary.csv')
display(summary[['pool','name','mean_f1','repeat_sd','macro_f1','delta_current_ani','repeat_wins','fold_wins','fold_ties']])
display(summary[['pool','name','f1_Safety','f1_Funding','f1_Efficacy','f1_Recruitment','f1_Other']])


,pool,name,mean_f1,repeat_sd,macro_f1,delta_current_ani,repeat_wins,fold_wins,fold_ties
0,train157,ani_current,0.921444,0.007355,0.872553,0.000000,0.0,0.0,15.0
1,train157,ani_legacy,0.918896,0.012325,0.855981,-0.002548,2.0,5.0,2.0
2,train157,trusted_two_pass,0.920594,0.008439,0.856218,-0.000849,1.0,5.0,5.0
3,trainval235,ani_current,0.913741,0.006297,0.836665,0.000000,0.0,0.0,40.0
4,trainval235,ani_legacy,0.908865,0.006631,0.822043,-0.004876,0.0,9.0,12.0
5,trainval235,trusted_two_pass,0.915337,0.004152,0.831522,0.001596,5.0,20.0,7.0


,pool,name,f1_Safety,f1_Funding,f1_Efficacy,f1_Recruitment,f1_Other
0,train157,ani_current,0.897386,0.906092,0.924307,0.956948,0.678030
1,train157,ani_legacy,0.899397,0.835275,0.925368,0.960017,0.659849
2,train157,trusted_two_pass,0.930905,0.829359,0.892661,0.967993,0.660173
3,trainval235,ani_current,0.913298,0.817210,0.872468,0.973269,0.607079
4,trainval235,ani_legacy,0.907372,0.760623,0.869526,0.972481,0.600213
5,trainval235,trusted_two_pass,0.920669,0.773221,0.867622,0.977536,0.618564


Our candidate wins only one of five repeat averages on157 (two ties, two losses). On235 it wins five of eight repeats and loses three. Fold comparisons are5 wins/5 ties/5 losses on157 and20 wins/7 ties/13 losses on235. These are small, mixed differences, not a consistent improvement across the two required pools. Do not compare absolute scores between the two pools: they contain different trial populations and different fitting-set sizes.

The additional diagnostic comparisons below also use the **current** Ani recipe. The candidate has higher sample F1 on the old validation split and the earliest/latest chronological checks, and ties the middle chronological check. This does not override the mixed CV evidence or Ani's higher macro F1 in both pools. All of these are reused development comparisons.


In [2]:
display(pd.read_csv(OUT/'paired_trial_bootstrap.csv'))
diagnostics=pd.read_csv(OUT/'current_diagnostic_scores.csv')
display(diagnostics[['split','name','fit_rows','unlabeled_rows','held_rows','sample_f1','macro_f1']])
display(json.loads((OUT/'checks.json').read_text()))


,pool,delta,lower_95,upper_95,caveat
0,train157,-0.000849,-0.011465,0.009342,Descriptive trial bootstrap; not adjusted for ...
1,trainval235,0.001596,-0.005142,0.008511,Descriptive trial bootstrap; not adjusted for ...


,split,name,fit_rows,unlabeled_rows,held_rows,sample_f1,macro_f1
0,original_validation,ani_current,157,1014,78,0.871795,0.742026
1,original_validation,trusted_two_pass,157,1014,78,0.880342,0.695565
2,after_2003,ani_current,126,60,109,0.811927,0.759425
3,after_2003,trusted_two_pass,126,60,109,0.824159,0.778561
4,after_2005,ani_current,149,302,86,0.860465,0.820562
5,after_2005,trusted_two_pass,149,302,86,0.860465,0.804538
6,after_2007,ani_current,198,724,37,0.864865,0.828648
7,after_2007,trusted_two_pass,198,724,37,0.887387,0.848936


{'fit_role_checks': 55,
 'all_saved_splits_used': True,
 'current_ani_fetched_commit': '8a8c3fca70f13b62e989739a0bc1d97d229186e1',
 'legacy_235_probabilities_reproduced_on_all40folds': True,
 'candidate_235_probabilities_reproduced_on_all40folds': True,
 'all_pooled_scores_independently_rescored': True,
 'no_recipe_tuning': True,
 'no_test_feature_fitting': True,
 'source_sha256': '9a1e9336f37da7433b8daf5f7166522488729689168e2cef418a0cd2cc82be42'}

## Checks and reproducibility

Every fit records its permitted/held-out IDs. All55 split-role checks pass. On all40 folds of the235-label pool, the legacy wrapper reproduces the historical Ani probabilities and the selected candidate reproduces its frozen probabilities. The157-label results are newly fitted on the saved15folds. All pooled scores are independently recalculated from persisted predictions.

The source below is the complete comparison driver. It uses the unchanged `trusted_solution.py` and repository metric helpers; Ani's complete source notebook is bundled alongside the results. Run `python part1/compare_ani_pools.py` from the repository environment to reproduce. The pinned source check intentionally refuses to silently reuse this result folder if Ani's fetched source changes. Use a new versioned audit for a new Ani revision.


In [3]:
import sys,types
from IPython.core.magic import register_cell_magic
sys.path.insert(0,str(PART1))
@register_cell_magic
def source_module(name,source):
    module=types.ModuleType(name.strip());module.__file__=str(PART1/(name.strip()+'.py'))
    sys.modules[name.strip()]=module;exec(compile(source,module.__file__,'exec'),module.__dict__)


In [4]:
%%source_module compare_ani_pools
"""Corrective audit: selected candidate vs fetched current Ani on both label pools."""
import ast
import json
import subprocess
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, normalize
from threadpoolctl import threadpool_limits

from simple_benchmark import ROOT, BASE, digest, metrics
from simple_models import FEATURES, REASONS, decide
from trusted_solution import TrustedTwoPassClassifier

OUT=ROOT/'experiments/current_ani_pool_audit'


def read_current():
    OUT.mkdir(exist_ok=True)
    commit=subprocess.check_output(['git','rev-parse','origin/ani'],cwd=ROOT,text=True).strip()
    notebook=subprocess.check_output(['git','show',f'{commit}:midterm/part1_best_model.ipynb'],cwd=ROOT)
    path=OUT/'ani_current_source.ipynb'
    if path.exists():assert path.read_bytes()==notebook,'Use a new audit folder for a changed Ani commit.'
    else:path.write_bytes(notebook)
    source='\n\n'.join(''.join(c['source']) for c in json.loads(notebook)['cells'] if c['cell_type']=='code')
    names={'decide','expand','multinomial_lr','knn','gmm_cluster_label','shrinkage_lda','base_model','final_model'}
    tree=ast.parse(source)
    functions='\n\n'.join(ast.get_source_segment(source,n) for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in names)
    assert len([n for n in tree.body if isinstance(n,ast.FunctionDef) and n.name in names])==len(names)
    (OUT/'ani_functions.py').write_text(functions+'\n')
    return commit,functions


def ani_predict(frame,fit,unknown,held,functions,legacy=False):
    X=frame[FEATURES].to_numpy()
    pool=np.sort(np.r_[fit,unknown])
    assert set(pool).isdisjoint(held)
    scaler=StandardScaler().fit(X[pool]);Z=scaler.transform(X)
    pca=PCA(30,random_state=0).fit(Z[pool])
    gmm=GaussianMixture(30,covariance_type='diag',n_init=3,random_state=0).fit(pca.transform(Z[pool]))
    assert gmm.converged_
    ns=dict(np=np,normalize=normalize,LogisticRegression=LogisticRegression,
            LinearDiscriminantAnalysis=LinearDiscriminantAnalysis,EMB=FEATURES,pca30=pca,gmm=gmm)
    exec(compile(functions,str(OUT/'ani_functions.py'),'exec'),ns)
    options=dict(w_lda=0.,C=.3) if legacy else {}
    return ns['final_model'](Z[fit],frame.iloc[fit][REASONS].to_numpy(int),Z[held],Z[unknown],**options)


def main():
    commit,functions=read_current()
    train=pd.read_csv(ROOT/'data/part1_train.csv');valid=pd.read_csv(ROOT/'data/part1_val.csv')
    unknown=np.flatnonzero(train[REASONS].lt(0).all(axis=1).to_numpy())
    pools=[('train157',train,ROOT/'experiments/classifier_cross_validation/splits',5,3),
           ('trainval235',pd.concat([train,valid],ignore_index=True),BASE/'splits',8,5)]
    protocol=dict(ani_commit=commit,ani_source='midterm/part1_best_model.ipynb',source_sha256=digest(OUT/'ani_current_source.ipynb'),
                  old_reference='no LDA, C0.3',current_ani='LDA0.2 + LR0.5 at C0.1 + KNN0.3; GMM-assisted2passes',
                  candidate_source_sha256=digest(ROOT/'trusted_solution.py'),
                  pools={name:dict(labels=157 if name=='train157' else 235,repeats=repeats,folds=nfold,
                                  split_sha256={f'repeat{r}.csv':digest(splitdir/f'repeat{r}.csv') for r in range(repeats)})
                         for name,_,splitdir,repeats,nfold in pools},
                  policy='Strict fold-local feature fitting for all models. Exclude held-out/test inputs. 157-pool fits also exclude all78validation inputs and labels.',
                  native_transductive_comparison_run=False,no_parameter_tuning=True,
                  correction='Prior claims used only235-label CV and an older Ani recipe; they did not establish superiority over current Ani on both label pools.')
    (OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
    scores=[];repeats=[];roles=[];prediction_frames=[]
    with threadpool_limits(limits=1),warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        for pool_name,frame,splitdir,nrepeats,nfold in pools:
            verified=np.flatnonzero(frame[REASONS].ge(0).all(axis=1).to_numpy())
            ids=frame.iloc[verified].ID
            for repeat in range(nrepeats):
                assignment=pd.read_csv(splitdir/f'repeat{repeat}.csv').set_index('ID').loc[ids,'fold'].to_numpy()
                repeated={name:np.full((len(verified),5),-1,int) for name in ['ani_current','ani_legacy','trusted_two_pass']}
                for fold in range(nfold):
                    held_mask=assignment==fold;held=verified[held_mask];fit=verified[~held_mask]
                    assert len(np.intersect1d(np.r_[fit,unknown],held))==0
                    probs={}
                    for name,legacy in [('ani_current',False),('ani_legacy',True)]:
                        probs[name]=ani_predict(frame,fit,unknown,held,functions,legacy)
                    model=TrustedTwoPassClassifier().fit(frame.iloc[fit][FEATURES].to_numpy(),frame.iloc[fit][REASONS].to_numpy(int),frame.iloc[unknown][FEATURES].to_numpy())
                    probs['trusted_two_pass']=model.predict_proba(frame.iloc[held][FEATURES].to_numpy())
                    if pool_name=='trainval235':
                        old=pd.read_csv(BASE/'predictions'/f'repeat{repeat}_fold{fold}__ani_train_only.csv').set_index('ID').loc[frame.iloc[held].ID]
                        np.testing.assert_allclose(probs['ani_legacy'],old[[f'probability_{r}' for r in REASONS]],atol=1e-10,rtol=1e-8)
                        frozen=np.load(ROOT/'experiments/trusted_iteration/06_final_regularization_check/probabilities.npz')['final_r2_control']
                        np.testing.assert_allclose(probs['trusted_two_pass'],frozen[repeat,held_mask],atol=1e-10,rtol=1e-8)
                    truth=frame.iloc[held][REASONS].to_numpy(int)
                    for name,P in probs.items():
                        pred=decide(P,.3);repeated[name][held_mask]=pred
                        scores.append(dict(pool=pool_name,repeat=repeat,fold=fold,name=name,fit_rows=len(fit),held_rows=len(held),**metrics(truth,pred)))
                        df=pd.DataFrame(dict(ID=frame.iloc[held].ID,pool=pool_name,repeat=repeat,fold=fold,name=name))
                        for j,r in enumerate(REASONS):df[f'actual_{r}']=truth[:,j];df[f'predicted_{r}']=pred[:,j];df[f'probability_{r}']=P[:,j]
                        prediction_frames.append(df)
                    roles.append(dict(pool=pool_name,repeat=repeat,fold=fold,fit_ids=frame.iloc[fit].ID.tolist(),held_ids=frame.iloc[held].ID.tolist(),unlabeled_ids=frame.iloc[unknown].ID.tolist(),test_features_used=False))
                for name,pred in repeated.items():
                    assert (pred>=0).all()
                    repeats.append(dict(pool=pool_name,repeat=repeat,name=name,**metrics(frame.iloc[verified][REASONS].to_numpy(int),pred)))
                print(pool_name,'repeat',repeat,{r['name']:round(r['sample_f1'],6) for r in repeats if r['pool']==pool_name and r['repeat']==repeat},flush=True)
        # Correct the four diagnostic comparisons to use the current Ani recipe.
        frame=pd.concat([train,valid],ignore_index=True);verified=np.flatnonzero(frame[REASONS].ge(0).all(axis=1).to_numpy())
        diagnostic=[]
        for label in ['original_validation','after_2003','after_2005','after_2007']:
            if label=='original_validation':fit=verified[verified<len(train)];held=verified[verified>=len(train)];u=unknown
            else:
                year=int(label.split('_')[-1]);early=frame.start_year.to_numpy()<=year;fit=verified[early[verified]];held=verified[~early[verified]];u=unknown[early[unknown]]
            P=ani_predict(frame,fit,u,held,functions)
            model=TrustedTwoPassClassifier().fit(frame.iloc[fit][FEATURES].to_numpy(),frame.iloc[fit][REASONS].to_numpy(int),frame.iloc[u][FEATURES].to_numpy())
            for name,prob in [('ani_current',P),('trusted_two_pass',model.predict_proba(frame.iloc[held][FEATURES].to_numpy()))]:
                pred=decide(prob,.3);truth=frame.iloc[held][REASONS].to_numpy(int)
                diagnostic.append(dict(split=label,name=name,fit_rows=len(fit),unlabeled_rows=len(u),held_rows=len(held),**metrics(truth,pred)))
        pd.DataFrame(diagnostic).to_csv(OUT/'current_diagnostic_scores.csv',index=False)
    repeat_table=pd.DataFrame(repeats);fold_table=pd.DataFrame(scores)
    repeat_table.to_csv(OUT/'repeat_scores.csv',index=False);fold_table.to_csv(OUT/'fold_scores.csv',index=False)
    pd.concat(prediction_frames).to_csv(OUT/'predictions.csv',index=False)
    (OUT/'role_audit.json').write_text(json.dumps(roles,separators=(',',':'))+'\n')
    summary=repeat_table.groupby(['pool','name']).agg(mean_f1=('sample_f1','mean'),repeat_sd=('sample_f1','std'),macro_f1=('macro_f1','mean'),**{f'f1_{r}':(f'f1_{r}','mean') for r in REASONS})
    for pool,_,_,_,_ in pools:
        ref=repeat_table[(repeat_table.pool==pool)&(repeat_table.name=='ani_current')].sample_f1.to_numpy()
        fr=fold_table[(fold_table.pool==pool)&(fold_table.name=='ani_current')].sample_f1.to_numpy()
        for name in ['ani_current','ani_legacy','trusted_two_pass']:
            diff=repeat_table[(repeat_table.pool==pool)&(repeat_table.name==name)].sample_f1.to_numpy()-ref
            fd=fold_table[(fold_table.pool==pool)&(fold_table.name==name)].sample_f1.to_numpy()-fr
            summary.loc[(pool,name),'delta_current_ani']=diff.mean()
            summary.loc[(pool,name),'repeat_wins']=sum(diff>1e-12)
            summary.loc[(pool,name),'fold_wins']=sum(fd>1e-12)
            summary.loc[(pool,name),'fold_ties']=sum(abs(fd)<1e-12)
    summary.to_csv(OUT/'summary.csv')
    print(summary.to_string(),flush=True)
    print(pd.DataFrame(diagnostic)[['split','name','sample_f1','macro_f1']].to_string(index=False),flush=True)


if __name__=='__main__':main()


In [5]:
RERUN_AUDIT=False
if RERUN_AUDIT:
    import compare_ani_pools
    compare_ani_pools.main()
else:
    print('Showing the saved, independently rescored current-Ani comparison on both pools.')


Showing the saved, independently rescored current-Ani comparison on both pools.
